# Find the Tents

_Combinatorial Optimization course, FEE CTU in Prague. Created by [Industrial Informatics Department](http://industrialinformatics.fel.cvut.cz)._

The problem was taken from https://www.brainbashers.com/tents.asp ; there, you can try to solve some examples manually.

## Task

Find all of the hidden tents in the forest grid.

You know that:

- Each tent is attached to one tree (so there are as many tents as there are trees).
- A tent can only be found horizontally or vertically adjacent to a tree.
- Tents are never adjacent to each other, neither vertically, horizontally, nor diagonally.
- A tree might be next to two tents but is only connected to one.

You are also given two vectors indicating how many tents are in each respective row or column of the forest grid.


## Input

You are given a positive integer $n \geq 2$, representing the size of the forest grid (assume it is a square of size $(n \times n$). You are also given vectors $\mathbf r = (r_1, \dots, r_n)$ and $\mathbf c = (c_1, \dots, c_n)$ representing the numbers of the tents in the rows and columns of the forest grid. Finally, you are given a list of coordinates of the trees $((x_1, y_1), \dots, (x_k, y_k))$.


In [1]:
!pip install gurobipy
import gurobipy as g

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [4]:
# 2x2 - Extra small (for debugging)
n = 3
r = (1, 1, 0)
c = (1, 0, 1)
trees = [(1,1), (3,2)]

In [ ]:
# 8x8 - Medium
n = 8
r = (3, 1, 1, 2, 0, 2, 0, 3)
c = (2, 1, 2, 2 ,1, 1 ,2 ,1)
trees = [(2, 1), (5, 1), (6, 1),
         (1, 2),
         (3, 3),
         (3, 4), (6, 4),
         (4, 5), (6, 5),
         (8, 7),
         (2, 8), (4, 8)]

In [ ]:
# Weekly special
n = 20
r = (7, 2, 3, 4, 3, 5, 4, 4, 4, 4, 3, 6, 3, 6, 2, 3, 6, 3, 3, 5)
c = (6, 4, 3, 5, 4, 4, 4, 3, 5, 3, 4, 3, 4, 4, 6, 3 ,4, 3, 6, 2)
trees = [(3, 1), (4, 1), (8, 1), (13, 1), (15, 1),
         (1, 2), (9, 2), (18, 2), (19, 2),
         (5, 3), (12, 3), (15, 3),
         (2, 4), (4, 4), (9, 4), (17, 4),
         (6, 5), (10, 5), (13, 5), (17, 5), (20, 5),
         (1, 6), (7, 6), (10, 6), (12, 6), (16, 6),
         (20, 7),
         (1, 8), (4, 8), (5, 8), (11, 8), (13, 8), (14, 8), (19, 8),
         (4, 9), (6, 9), (9, 9), (15, 9), (17, 9),
         (8, 10), (17, 10), (19, 10),
         (12, 11),
         (5, 12), (7, 12), (14, 12), (16, 12),
         (1, 13), (2, 13), (6, 13), (19, 13),
         (11, 14), (14, 14), (20, 14),
         (3, 15), (5, 15), (6, 15), (8, 15), (13, 15), (20, 15),
         (2, 16), (3, 16), (10, 16),
         (8, 17), (11, 17), (14, 17), (15, 17),
         (2, 18), (6, 18), (9, 18), (12, 18), (13, 18), (18, 18),
         (2, 19), (7, 19), (15, 19), (17, 19), (20, 19),
         (5, 20), (10, 20)]

## Output

You should find the coordinates $(x_i, y_i), i \in \{1,\dots,k\}$, of the individual tents.

## Model

In [ ]:
tree_grid = [[False for _ in range(n+2)] for _ in range(n+2)]
for t_x, t_y in trees:
    tree_grid[t_y][t_x] = True

# TODO
model = g.Model()

#pocet stanu 
#x = model.addVar( vtype=g.GRB.INTEGER, name="x", lb=0)

#vektor bin promennych je tam stan neni tam stan
t = model.addVars(
    range(n),
    range(n),

    vtype=g.GRB.BINARY,
    name="t"
)

trees0 = trees.copy()
for i in range (len(trees)):
    x, y = trees0[i]
    trees0[i] = (x-1,y-1)
 
#print(trees0)    

for x, y in trees0:
    model.addConstr(t[x,y] == 0)
    
#for x, y in trees:
 #   model.addConstr(t[x+1, y] + t[x-1, y] + t[x, y+1] + t[x, y-1] >= 1)
 
for x, y in trees0:
    sousedi = []
     
     
    if (x+1 < n ):
        sousedi.append(t[x+1, y])
    if (x-1 >= 0):
        sousedi.append(t[x-1, y])
    if (y+1 < n):
        sousedi.append(t[x, y+1])
    if (y-1 >= 0):
        sousedi.append(t[x, y-1])
         
    model.addConstr(g.quicksum(sousedi[i] for i in range (len(sousedi)) )>= 1)
        
for row in range (n):
    model.addConstr(g.quicksum(t[x, row] for x in range (n)) == r[row])

for col in range(n):
    model.addConstr(g.quicksum(t[col, y] for y in range(n))  == c[col])
    
for row in range (n):
    for col in range (n):
        if (t[row, col] == 1):
            sousedi = []    
            
            if (x+1 < n ):
                    sousedi.append(t[x+1, y])
            if (x-1 >= 0):
                    sousedi.append(t[x-1, y])
            if (y+1 < n):
                    sousedi.append(t[x, y+1])
            if (y-1 >= 0):
                    sousedi.append(t[x, y-1])
                    
            if(x+1 < n and y+1 < n):
                sousedi.append(t[x+1, y+1])
            if(x-1 >= 0 and y+1 < n):
                sousedi.append(t[x-1,y+1])
            if(x-1 >= 0 and y-1 >= 0):
                sousedi.append(t[x-1,y-1])
            if(x+1 < n and y-1 >= 0):
                sousedi.append(t[x+1,y-1])
            
            model.addConstr(g.quicksum(sousedi) == 0)

    
#tady se nic nemusi maximalizovat
#model.setObjective(g.quicksum(x[i] for i in range n), )


TypeError: 'tupledict' object is not callable

 ##  Visualization

In [2]:
import matplotlib.pyplot as plt
import numpy as np

def visualize(n, trees, tents, r, c):
    grid = [["." for _ in range(n+2)] for _ in range(n+2)]

    for t_x, t_y in tents:
        grid[t_y][t_x] = "X"

    for t_x, t_y in trees:
        grid[t_y][t_x] = "T"

    print("  ", end="")
    for c_cur in c:
        print(c_cur, end=" ")
    print()

    for y in range(1, n+1):
        print(r[y-1], end=" ")
        for x in range(1, n+1):
            print(grid[y][x], end=" ")

        print()

/home/jirkadolezal/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [ ]:
visualize(n, trees, tents, r, c)